# F1 · Hold-out, 5-fold, 10-fold, repeated and leave-one-out: bias and variance of the estimate

**Type**: Compare · **Level**: ●●● challenge · **Time**: about 50 minutes · **extended path**

## Objective

Compare five validation strategies on 30 simulated datasets of 200 rows, measuring for each how much the accuracy estimate departs
from the **true** value (computed on 4000 new rows): bias, variability and mean squared error.

## Dataset

`generate(seed)`: with `make_classification` (10 features, 4 informative) a dataset of 200 rows and a **truth** one of 4000. The true performance
is the accuracy of the model trained on the 200 rows and evaluated on the 4000. Model: logistic regression with scaler.

## Educational purpose

Replace "10-fold is better" with numbers. Every strategy is a compromise between how much data it uses to train (bias), how many folds it
evaluates (variance) and how much it costs (trainings).

## Task

1. Write `estimate(strategy, X, y, seed)`: apply the strategy to `(X, y)` and return the estimated accuracy. Use `SCHEMES`, a dictionary name → function `seed → splitter` (already defined): a `ShuffleSplit` that generates a single split is a hold-out.
2. For each `seed` from 0 to 29: generate the data, train the model on the 200 rows and compute `true` (accuracy on the 4000); then compute the estimate of each strategy with `cross_val_score(model, X, y, cv=splitter, scoring="accuracy").mean()`. Collect in `errors` (30 × 5 DataFrame) the error `estimate − true`.
3. Build `table` (rows: strategies; columns `bias` = mean of the errors, `std_dev` = standard deviation of the errors, `rmse` = root of the mean of the squares) and `n_trainings` (number of fits per strategy on one dataset).
4. Assign to `most_variable` the name of the strategy with the largest `std_dev` and to `best_rmse` the one with the smallest `rmse`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (LeaveOneOut, RepeatedStratifiedKFold, ShuffleSplit, StratifiedKFold, cross_val_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))


def generate(seed):
    X, y = make_classification(n_samples=4200, n_features=10, n_informative=4, class_sep=0.8, flip_y=0.05, random_state=seed)
    return X[:200], y[:200], X[200:], y[200:]


SCHEMES = {
    "hold-out 25%": lambda s: ShuffleSplit(1, test_size=0.25, random_state=s),
    "5-fold": lambda s: StratifiedKFold(5, shuffle=True, random_state=s),
    "10-fold": lambda s: StratifiedKFold(10, shuffle=True, random_state=s),
    "repeated 5x5": lambda s: RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=s),
    "leave-one-out": lambda s: LeaveOneOut(),
}

## Your work

In [ ]:
rows = []
for seed in range(30):
    X, y, X_true, y_true = generate(seed)
    true = ...
    seed_errors = {}
    for name, make in SCHEMES.items():
        ...
    rows.append(seed_errors)
errors = pd.DataFrame(rows)
errors.round(3).head(3)

In [ ]:
table = pd.DataFrame({"bias": ..., "std_dev": ..., "rmse": ...})
n_trainings = {"hold-out 25%": ..., "5-fold": ..., "10-fold": ..., "repeated 5x5": ..., "leave-one-out": ...}
most_variable = ...
best_rmse = ...
table.round(4)

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`true = model.fit(X, y).score(X_true, y_true)` (accuracy). For every strategy: `cross_val_score(model, X, y, cv=make(seed), scoring="accuracy").mean() - true`. `table = pd.DataFrame({"bias": errors.mean(), "std_dev": errors.std(), "rmse": np.sqrt((errors ** 2).mean())})`.'):
    check(errors.shape == (30, 5) and list(errors.columns) == list(SCHEMES), "30 datasets x 5 strategies", "one error estimate - true for every dataset and strategy")
    check(list(table.columns) == ["bias", "std_dev", "rmse"] and set(table.index) == set(SCHEMES), "table with the three metrics per strategy", "bias, std_dev, rmse")
    check(close(table["bias"].to_numpy(), errors.mean().reindex(table.index).to_numpy()) and close(table["rmse"].to_numpy(), np.sqrt((errors ** 2).mean()).reindex(table.index).to_numpy()),
          "bias and rmse computed correctly", "bias = mean, rmse = root of the mean of the squares")
    check(n_trainings == {"hold-out 25%": 1, "5-fold": 5, "10-fold": 10, "repeated 5x5": 25, "leave-one-out": 200},
          "correct count of trainings", "hold-out 1, 5-fold 5, 10-fold 10, repeated 25, leave-one-out 200")
    check(most_variable == table["std_dev"].idxmax() and most_variable == "hold-out 25%", "the hold-out is the most variable strategy", "a single split with 50 validation rows is very noisy")
    check(best_rmse == table["rmse"].idxmin() and best_rmse != "hold-out 25%", "the strategy with the smallest error is not the hold-out", "look at the rmse column")
    check(table.loc["hold-out 25%", "rmse"] > 1.3 * table.loc["repeated 5x5", "rmse"],
          "the hold-out has a mean squared error over 30% larger than the repeated validation", "repeating the validation reduces the variability a lot")
done()

---
The commented solution is in `solutions/F_compare/F1_validation_strategies_bias_and_variance.ipynb`. Open it after you pass the check, or after you have really tried.